# Scalability Benchmark

Measures wall-clock computation time for **SBM** (Stochastic Block Model), **CM** (Configuration Model), and **GM** (Geometric Model) graphs as a function of graph size `n`.

**Experimental setup:**
- `n` ∈ {100, 250, 500, 750, 1000, 1250, 1500} nodes
- 5 parameter values per graph model
- 25 graphs per `(n, param)` pair
- Timed pipeline stages: graph generation → EDRep embedding → exact Wasserstein-2 → Monte Carlo Wasserstein-2

**Checkpointing:** results are saved to `output_scalability/` as NPZ files. Re-running any phase cell is safe — existing checkpoints are detected and skipped automatically.


In [ ]:
import numpy as np
import random
import itertools
import pandas as pd
import os
import time
import ot
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import squareform
from sklearn.metrics.pairwise import euclidean_distances
from EDRep_main.EDRep import NodeEmbedding
from tqdm.notebook import tqdm
from functions import *
from graph_generators import *

---
## Configuration

Shared parameters for the benchmark: graph sizes, number of graphs per condition, embedding dimension, and per-model parameter grids.

In [ ]:
# ── Graph size sweep ──────────────────────────────────────────────────────────
n_nodes_list   = [100, 250, 500, 750, 1000, 1250, 1500]
n_graphs_bench = 25      # graphs per (n, param) pair
dim            = 16      # EDRep embedding dimension
symmetric      = True
make_connected = True
c              = 10      # target average degree

# ── Per-model parameter grids ─────────────────────────────────────────────────
# alpha controls community separability (SBM) or degree heterogeneity (CM);
# beta controls the geometric connection decay rate (GM).
sbm_alphas_bench = [0.85, 1.04, 1.28, 1.58, 1.75]
cm_alphas_bench  = [0.70, 1.06, 1.45, 1.61, 1.98]
gm_betas_bench   = [1.40, 1.64, 1.92, 2.25, 2.63]

# ── Output directory ──────────────────────────────────────────────────────────
out_dir_bench = "output_scalability"
os.makedirs(out_dir_bench, exist_ok=True)
print("Configuration ready.")

---
## Timing wrappers

Each wrapper runs one pipeline stage and returns **both the result and the elapsed wall-clock time** (in seconds, measured with `time.perf_counter`).

| Wrapper | Stage |
|---|---|
| `timed_gen_sbm` | DCSBM edge-list generation for the SBM |
| `timed_gen_cm` | CM edge-list generation via `gen_cm_via_dcsbm` |
| `timed_gen_gm` | Geometric Model edge-list generation |
| `timed_embed` | EDRep `NodeEmbedding` call (graph → embedding matrix) |
| `timed_w2` | Exact Wasserstein-2 over all C(25, 2) graph pairs, using sorted upper-triangular scalar products of the embedding |
| `timed_w2_sampled` | Monte Carlo Wasserstein-2: samples 10 % of directed node pairs per graph, fully vectorised (no Python loop over pairs) |

`_w2_from_sorted` is a shared helper that computes the W2 distance matrix from a list of pre-sorted 1-D distributions.

In [ ]:
def timed_gen_sbm(n, alpha):
    k2     = 2
    l2     = np.array([i // (n // k2) for i in range(n)], dtype=int)
    theta2 = np.ones(n)
    c_in, c_out = compute_cin_cout(c, k2, alpha)
    C2 = np.ones((k2, k2)) * c_out + np.diag(np.ones(k2)) * (c_in - c_out)
    t0 = time.perf_counter()
    df = DCSBM((C2, c, l2, theta2, symmetric, make_connected))
    return df, time.perf_counter() - t0


def timed_gen_cm(n, alpha):
    t0 = time.perf_counter()
    df = gen_cm_via_dcsbm((n, c, alpha, symmetric, make_connected))
    return df, time.perf_counter() - t0


def timed_gen_gm(n, beta):
    t0    = time.perf_counter()
    r_xy  = np.random.uniform(0, 1, n)
    th    = np.random.uniform(0, 2 * np.pi, n)
    X_pos = np.column_stack([r_xy * np.cos(th), r_xy * np.sin(th)])
    df    = GeometricModel((X_pos, c, beta))
    return df, time.perf_counter() - t0


def timed_embed(A, dim_):
    t0 = time.perf_counter()
    X  = node_embedding(A, dim_)
    return X, time.perf_counter() - t0


def _w2_from_sorted(S_list):
    """Wasserstein-2 distance matrix from a list of sorted scalar-product arrays."""
    pairs = list(itertools.combinations(range(len(S_list)), 2))
    vals  = []
    for i, j in pairs:
        s1, s2 = S_list[i], S_list[j]
        if len(s1) == len(s2):
            vals.append(float(np.sqrt(np.mean((s1 - s2) ** 2))))
        else:
            vals.append(float(ot.wasserstein_1d(s1, s2, p=2) ** 0.5))
    return squareform(vals)


def timed_w2(emb_list):
    """Exact W2: full embedding, sorts all C(n,2) upper-triangular scalar products."""
    t0     = time.perf_counter()
    S_list = []
    for X in emb_list:
        X32 = X.astype(np.float32)
        iu  = np.triu_indices(len(X32), k=1)
        S_list.append(np.sort((X32 @ X32.T)[iu]))
    D = _w2_from_sorted(S_list)
    return D, time.perf_counter() - t0


def timed_w2_sampled(emb_list, sample_frac=0.1):
    """MC W2: full embedding, then sample 10% of directed pairs (i≠j).
    Fully vectorised — no Python loops over pairs."""
    t0     = time.perf_counter()
    S_list = []
    for X in emb_list:
        n_nodes = len(X)
        n_mc    = max(10, int(sample_frac * n_nodes * (n_nodes - 1)))
        # oversample by 5% to cover pairs where i==j after masking
        over    = int(n_mc * 1.05) + 10
        a = np.random.randint(0, n_nodes, over)
        b = np.random.randint(0, n_nodes, over)
        mask = a != b
        a, b = a[mask][:n_mc], b[mask][:n_mc]
        X32  = X.astype(np.float32)
        d    = np.einsum('ij,ij->i', X32[a], X32[b])
        S_list.append(np.sort(d))
    D = _w2_from_sorted(S_list)
    return D, time.perf_counter() - t0


model_configs = [
    ("sbm", sbm_alphas_bench, timed_gen_sbm),
    ("cm",  cm_alphas_bench,  timed_gen_cm),
    ("gm",  gm_betas_bench,   timed_gen_gm),
]


---
## Phase 1 — Graph generation and embedding

Iterates over all `(model, param, n)` triples. For each combination:
1. Generates `n_graphs_bench = 25` graphs using the appropriate timed generator.
2. Converts each edge-list DataFrame to a sparse CSR adjacency matrix and computes its EDRep embedding.
3. Saves embeddings and per-graph timings to a compressed NPZ checkpoint in `output_scalability/`.

**Checkpointing:** a file is skipped if it already contains the `embs` key.

**NPZ contents per file:**

| Key | Shape | Description |
|---|---|---|
| `embs` | `(25, n_nodes, dim)` | float32 embedding matrices |
| `gen_times` | `(25,)` | wall-clock seconds per graph generation |
| `emb_times` | `(25,)` | wall-clock seconds per embedding |

In [ ]:
for model_name, params, gen_fn in model_configs:
    print(f"\n=== {model_name.upper()} ===")
    for p in tqdm(params, desc=model_name):
        for n in n_nodes_list:
            npz_path = os.path.join(
                out_dir_bench, f"{model_name}_p{p:.2f}_n{n}.npz"
            )
            # skip if embeddings already saved
            if os.path.exists(npz_path):
                data = np.load(npz_path)
                if "embs" in data:
                    print(f"  skip  {os.path.basename(npz_path)}")
                    continue

            embs, gen_times, emb_times = [], [], []
            for _ in range(n_graphs_bench):
                df, tg = gen_fn(n, p)
                X, te  = timed_embed(df_to_adj(df, n), dim)
                embs.append(X.astype(np.float32))
                gen_times.append(tg)
                emb_times.append(te)

            np.savez_compressed(
                npz_path,
                embs      = np.stack(embs),          # (25, n_nodes, dim)
                gen_times = np.array(gen_times),      # (25,)
                emb_times = np.array(emb_times),      # (25,)
            )
            print(f"  saved {os.path.basename(npz_path)}  "
                  f"(mean emb={np.mean(emb_times):.3f}s)")


---
## Phase 2 — Distance computation

Loads each NPZ produced by Phase 1 and computes two pairwise distance matrices over the 25 graphs:

- **Exact W2** (`timed_w2`): builds the full upper-triangular scalar-product matrix `X @ Xᵀ`, sorts it, and computes the empirical Wasserstein-2 distance for every C(25, 2) pair.
- **Monte Carlo W2** (`timed_w2_sampled`): samples 10 % of directed node pairs per graph and computes W2 on the resulting 1-D distributions. Fully vectorised — no Python loop over graph pairs.

Timing and distance matrices are written back into the same NPZ file under new keys:

| Key | Description |
|---|---|
| `t_w2` | Total elapsed time for exact W2 (scalar, seconds) |
| `D_w2` | Exact W2 distance matrix, shape `(25, 25)` |
| `t_w2_sampled` | Total elapsed time for MC W2 (scalar, seconds) |
| `D_w2_sampled` | MC W2 distance matrix, shape `(25, 25)` |

**Checkpointing:** a file is skipped if it already contains both `t_w2` and `t_w2_sampled`.

In [ ]:
for model_name, params, _ in model_configs:
    print(f"\n=== {model_name.upper()} ===")
    for p in tqdm(params, desc=model_name):
        for n in n_nodes_list:
            npz_path = os.path.join(
                out_dir_bench, f"{model_name}_p{p:.2f}_n{n}.npz"
            )
            if not os.path.exists(npz_path):
                print(f"  missing embeddings for {os.path.basename(npz_path)} — run Phase 1 first")
                continue

            data = dict(np.load(npz_path))

            # skip if both distances already computed
            if "t_w2" in data and "t_w2_sampled" in data:
                print(f"  skip  {os.path.basename(npz_path)}")
                continue

            embs = [data["embs"][i] for i in range(n_graphs_bench)]

            if "t_w2" not in data:
                D_w, t_w = timed_w2(embs)
                data["t_w2"] = np.float64(t_w)
                data["D_w2"] = D_w

            if "t_w2_sampled" not in data:
                D_ws, t_ws = timed_w2_sampled(embs)
                data["t_w2_sampled"] = np.float64(t_ws)
                data["D_w2_sampled"] = D_ws

            np.savez_compressed(npz_path, **data)
            print(f"  saved {os.path.basename(npz_path)}  "
                  f"(w2={data['t_w2']:.4f}s  mc={data['t_w2_sampled']:.4f}s)")


---
## Aggregate results

Reads all NPZ checkpoints from `output_scalability/` and assembles a tidy DataFrame `df_bench`. Each row is one `(model, param, n)` combination.

| Column | Description |
|---|---|
| `model` | Graph model name (`sbm`, `cm`, `gm`) |
| `param` | Model parameter (`α` for SBM/CM, `β` for GM) |
| `n` | Number of nodes |
| `t_gen` | Mean graph generation time (s) |
| `t_emb` | Mean embedding time (s) |
| `t_w2` | Total exact W2 computation time (s) |
| `t_w2_sampled` | Total MC W2 computation time (s) |
| `mean_d_w2` | Mean pairwise exact W2 distance |
| `mean_d_w2_sampled` | Mean pairwise MC W2 distance |

In [ ]:
rows = []
for model_name, params, _ in model_configs:
    for p in params:
        for n in n_nodes_list:
            npz_path = os.path.join(
                out_dir_bench, f"{model_name}_p{p:.2f}_n{n}.npz"
            )
            if not os.path.exists(npz_path):
                continue
            data = np.load(npz_path)
            if "t_w2" not in data or "t_w2_sampled" not in data:
                continue
            D_w  = data["D_w2"]
            D_ws = data["D_w2_sampled"]
            rows.append({
                "model":             model_name,
                "param":             p,
                "n":                 n,
                "t_gen":             float(np.mean(data["gen_times"])),
                "t_emb":             float(np.mean(data["emb_times"])),
                "t_w2":              float(data["t_w2"]),
                "t_w2_sampled":      float(data["t_w2_sampled"]),
                "mean_d_w2":         float(np.mean(D_w[D_w > 0])),
                "mean_d_w2_sampled": float(np.mean(D_ws[D_ws > 0])),
            })
df_bench = pd.DataFrame(rows)

---
## Plots — Distance computation time vs. graph size

Compares **exact Wasserstein-2** and **Monte Carlo Wasserstein-2** computation time as a function of `n`, for each of the three graph models displayed side by side. The y-axis is shared across subplots to facilitate direct comparison.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5),sharey=True)
for ax, (model_name, _, __) in zip(axes, model_configs):
    sub = df_bench[df_bench.model == model_name]
    for time_col, label, color, marker, ls in [
        ("t_w2",        "Wasserstein exact", "#DD6722",   "s", "--"),
        ("t_w2_sampled","Wasserstein MC","#278644", "o", "-"),
    ]:
        grp = sub.groupby("n")[time_col]
        mean, std = grp.mean(), grp.std()
        ns = mean.index.tolist()
        ax.plot(ns, mean.values, marker=marker, color=color,
                linewidth=2.5, markersize=10, linestyle=ls, label=label)
        ax.fill_between(ns,
                        (mean ).values,
                        (mean ).values,
                        alpha=0.15, color=color)
    ax.set_title(model_name.upper(), fontsize=22, fontweight="bold")
    ax.set_xlabel("Nodes", fontsize=17, fontweight="bold")
    axes[0].set_ylabel("Computation time ", fontsize=17, fontweight="bold")
    ax.set_xticks(n_nodes_list)
    ax.tick_params(labelsize=14)
    ax.legend(frameon=False, prop={"size": 17, "weight": "bold"})
    ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()